In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from pathlib import Path
from sklearn.model_selection import train_test_split
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from sklearn.model_selection import KFold
from chemprop import data, featurizers, models, nn
from rdkit.Chem import MolFromSmiles
import concurrent.futures
import logging
from chemprop.data import datapoints, dataloader, MoleculeDataset
from chemprop.featurizers import SimpleMoleculeMolGraphFeaturizer
from chemprop.data.datapoints import MoleculeDatapoint
import torch
import chemprop.nn.metrics as chem_metrics
from chemprop.nn.agg import (
    MultiHeadAttentiveAggregation,
    GatedAttentiveAggregation,
    GatedAttentiveAggregationv2
)
from assets.functionchem import *
from assets.chempropcombination import *

df = pd.read_csv("./data/caco2data.csv")
df = df.rename(columns={"target": "Caco2Papp"}).copy()

# Ensure numeric target
df["Caco2Papp"] = pd.to_numeric(df["Caco2Papp"], errors="coerce")
df = df.dropna(subset=["Caco2Papp", "smiles"])
# Apply the scaffold computation for each SMILES
df['scaffold'] = df['smiles'].apply(compute_scaffold)

# Step 1: Get the unique scaffolds
scaffold_list = df['scaffold'].unique()

# Step 2: Split the list of scaffolds into train, test, and validation
train_scaffolds, temp_scaffolds = train_test_split(scaffold_list, test_size=0.2, random_state=42)
test_scaffolds, val_scaffolds = train_test_split(temp_scaffolds, test_size=0.5, random_state=42)

# Step 3: Create the train, test, and validation sets by filtering the original DataFrame based on scaffold
train_df = df[df['scaffold'].isin(train_scaffolds)]
test_df = df[df['scaffold'].isin(test_scaffolds)]
val_df = df[df['scaffold'].isin(val_scaffolds)]

# Step 4: Verify the distribution of scaffolds in each set
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Validation set size: {len(val_df)}")

results_df = run_chemprop_mp_agg_benchmark(
    train_df,
    val_df,
    test_df,
    target_column="Caco2Papp",
    max_epochs=200
)

Seed set to 42


Training set size: 2396
Test set size: 299
Validation set size: 300


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/tanush/anaconda3/envs/chemprop/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/tanush/Documents/WB_PBPK_MODEL_FINALIZED/submissionGit18MARCH_FinalizedCode/checkpoint/checkpoints_variants/Caco2Papp/BondMP3_Mean exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.



Training: BondMP3 + Mean


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

...

Output()

`Trainer.fit` stopped: `max_epochs=200` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Test: {'Target': 'Caco2Papp', 'MessagePassing': 'AtomMP6', 'Aggregation': 'MultiHead12', 'Test_MAE': 0.3009583353996277, 'Test_RMSE': 0.414156973361969, 'Test_R2': 0.978558361530304}

Final comparison:
       Target MessagePassing     Aggregation  Test_MAE  Test_RMSE   Test_R2
15  Caco2Papp        BondMP5      MultiHead4  0.282124   0.497234  0.969094
7   Caco2Papp        BondMP4  GatedAttentive  0.288241   0.484302  0.970680
22  Caco2Papp        BondMP6      MultiHead8  0.293009   0.453947  0.974240
4   Caco2Papp        BondMP3      MultiHead8  0.295671   0.490862  0.969881
29  Caco2Papp        AtomMP3     MultiHead12  0.297923   0.410013  0.978985
26  Caco2Papp        AtomMP3      Attentive1  0.298691   0.435214  0.976323
19  Caco2Papp        BondMP6  GatedAttentive  0.298833   0.482399  0.970910
42  Caco2Papp        AtomMP6            Mean  0.298931   0.515011  0.966844
47  Caco2Papp        AtomMP6     MultiHead12  0.300958   0.414157  0.978558
34  Caco2Papp        AtomMP4      Mult